In [10]:
import os

import numpy as np
from mahotas.thresholding import bernsen
from scipy.ndimage import median_filter
from skimage.filters import threshold_local, threshold_niblack, threshold_sauvola
from skimage.io import imread
from skimage.morphology import binary_opening, remove_small_objects

thresholding = [threshold_local, threshold_sauvola, threshold_niblack, bernsen]
preprocessing = [median_filter]
postprocessing = [remove_small_objects, binary_opening]


def dice_score(pred, gt): # like F1 score
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    return 2 * intersection / (pred.sum() + gt.sum() + 1e-8)

def iou_score(pred, gt): # intersect over unioni
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return intersection / (union + 1e-8)

In [ ]:
results = []

window_sizes = [5, 7, 11, 15, 21, 31, 41, 61, 91, 121, 151, 191]
k_values = [0.1, 0.2, 0.3, 0.4, 0.5, 1, 2, 3, 5, 7, 11]
r_values = [0.25, 0.5, 0.75, 1.0]
for window in window_sizes:
    for k in k_values:
        for r in r_values:
            dice_scores = []
            iou_scores = []

            for patch_num in range(1, 11):
                gt_dirpath = f'./ear_vessel_data/test/groundtruth/patch_{patch_num}'
                img_dirpath = f'./ear_vessel_data/test/image/patch_{patch_num}'
                file_list = os.listdir(gt_dirpath)

                for image_num in range(1, len(file_list)+1):
                    img_path = f"{image_num}.png"
                    img = imread(os.path.join(img_dirpath, img_path), as_gray=True) / 255.0
                    gt = imread(os.path.join(gt_dirpath, img_path), as_gray=True) / 255
                    gt = (gt > 0.5).astype(np.uint8)

                    # Apply Sauvola threshold
                    thresh = threshold_sauvola(img, window_size=window, k=k, r=r)
                    binary = (img > thresh).astype(np.uint8)

                    # Evaluate
                    dice = dice_score(binary, gt)
                    iou = iou_score(binary, gt)

                    dice_scores.append(dice)
                    iou_scores.append(iou)

            results.append({
                'window': window,
                'k': k,
                'r': r,
                'mean_dice': np.mean(dice_scores),
                'mean_iou': np.mean(iou_scores)
            })

# Sort and print top configs
top = sorted(results, key=lambda x: -x['mean_dice'])[:10]
for r in top:
    print(f"Window={r['window']}, k={r['k']}, r={r['r']}, -> Dice={r['mean_dice']:.4f}, IoU={r['mean_iou']:.4f}")
# Window=121, k=0.1, r=0.25, -> Dice=0.6821, IoU=0.5373
# Window=121, k=0.2, r=0.25, -> Dice=0.6820, IoU=0.5371
# Window=121, k=0.3, r=0.25, -> Dice=0.6819, IoU=0.5369
# Window=121, k=0.4, r=0.25, -> Dice=0.6817, IoU=0.5368
# Window=121, k=0.1, r=0.5, -> Dice=0.6817, IoU=0.5367
# Window=121, k=0.5, r=0.25, -> Dice=0.6816, IoU=0.5367
# Window=121, k=0.1, r=0.75, -> Dice=0.6816, IoU=0.5366
# Window=121, k=0.1, r=1.0, -> Dice=0.6815, IoU=0.5365
# Window=151, k=0.1, r=0.25, -> Dice=0.6813, IoU=0.5361
# Window=121, k=1, r=0.25, -> Dice=0.6812, IoU=0.5361

Window=121, k=0.1, r=0.25, -> Dice=0.6821, IoU=0.5373
Window=121, k=0.2, r=0.25, -> Dice=0.6820, IoU=0.5371
Window=121, k=0.3, r=0.25, -> Dice=0.6819, IoU=0.5369
Window=121, k=0.4, r=0.25, -> Dice=0.6817, IoU=0.5368
Window=121, k=0.1, r=0.5, -> Dice=0.6817, IoU=0.5367
Window=121, k=0.5, r=0.25, -> Dice=0.6816, IoU=0.5367
Window=121, k=0.1, r=0.75, -> Dice=0.6816, IoU=0.5366
Window=121, k=0.1, r=1.0, -> Dice=0.6815, IoU=0.5365
Window=151, k=0.1, r=0.25, -> Dice=0.6813, IoU=0.5361
Window=121, k=1, r=0.25, -> Dice=0.6812, IoU=0.5361


In [ ]:
results = []

window_sizes = [5, 7, 11, 15, 21, 31, 41, 61, 91, 121, 151, 191, 221, 255]
k_values = [0.1, 0.2, 0.3, 0.4, 0.5, 1, 2, 3, 5, 7, 11]
for window in window_sizes:
    for k in k_values:
        dice_scores = []
        iou_scores = []

        for patch_num in range(1, 11):
            gt_dirpath = f'./ear_vessel_data/test/groundtruth/patch_{patch_num}'
            img_dirpath = f'./ear_vessel_data/test/image/patch_{patch_num}'
            file_list = os.listdir(gt_dirpath)

            for image_num in range(1, len(file_list)+1):
                img_path = f"{image_num}.png"
                img = imread(os.path.join(img_dirpath, img_path), as_gray=True) / 255.0
                print(img.size())
                gt = imread(os.path.join(gt_dirpath, img_path), as_gray=True) / 255
                gt = (gt > 0.5).astype(np.uint8)

                # Apply Sauvola threshold
                thresh = threshold_niblack(img, window_size=window, k=k)
                binary = (img > thresh).astype(np.uint8)

                # Evaluate
                dice = dice_score(binary, gt)
                iou = iou_score(binary, gt)

                dice_scores.append(dice)
                iou_scores.append(iou)

        results.append({
            'window': window,
            'k': k,
            'mean_dice': np.mean(dice_scores),
            'mean_iou': np.mean(iou_scores)
        })

# Sort and print top configs
top = sorted(results, key=lambda x: -x['mean_dice'])[:10]
for r in top:
    print(f"Window={r['window']}, k={r['k']}, -> Dice={r['mean_dice']:.4f}, IoU={r['mean_iou']:.4f}")
# Window=191, k=0.1, r=0.25, -> Dice=0.6398, IoU=0.4992
# Window=151, k=0.1, r=0.25, -> Dice=0.6160, IoU=0.4757
# Window=121, k=0.1, r=0.25, -> Dice=0.5958, IoU=0.4553
# Window=91, k=0.1, r=0.25, -> Dice=0.5684, IoU=0.4275
# Window=61, k=0.1, r=0.25, -> Dice=0.5276, IoU=0.3873
# Window=5, k=0.2, r=0.25, -> Dice=0.5163, IoU=0.3586
# Window=7, k=0.1, r=0.25, -> Dice=0.5149, IoU=0.3579
# Window=191, k=0.2, r=0.25, -> Dice=0.5147, IoU=0.3810
# Window=5, k=0.1, r=0.25, -> Dice=0.5037, IoU=0.3465
# Window=151, k=0.2, r=0.25, -> Dice=0.4868, IoU=0.3557

Window=255, k=0.1, r={'window': 121, 'k': 1, 'r': 0.25, 'mean_dice': np.float64(0.6812097623189778), 'mean_iou': np.float64(0.5360725042202565)}, -> Dice=0.6632, IoU=0.5197
Window=221, k=0.1, r={'window': 121, 'k': 1, 'r': 0.25, 'mean_dice': np.float64(0.6812097623189778), 'mean_iou': np.float64(0.5360725042202565)}, -> Dice=0.6557, IoU=0.5133
Window=191, k=0.1, r={'window': 121, 'k': 1, 'r': 0.25, 'mean_dice': np.float64(0.6812097623189778), 'mean_iou': np.float64(0.5360725042202565)}, -> Dice=0.6398, IoU=0.4992
Window=151, k=0.1, r={'window': 121, 'k': 1, 'r': 0.25, 'mean_dice': np.float64(0.6812097623189778), 'mean_iou': np.float64(0.5360725042202565)}, -> Dice=0.6160, IoU=0.4757
Window=121, k=0.1, r={'window': 121, 'k': 1, 'r': 0.25, 'mean_dice': np.float64(0.6812097623189778), 'mean_iou': np.float64(0.5360725042202565)}, -> Dice=0.5958, IoU=0.4553
Window=91, k=0.1, r={'window': 121, 'k': 1, 'r': 0.25, 'mean_dice': np.float64(0.6812097623189778), 'mean_iou': np.float64(0.5360725042

In [ ]:
results = []

radius_values = [5, 10, 15, 25, 35]
contrast_threshold_values = [0.02, 0.04, 0.08, 0.12, 0.16]
gthresh_values = [None, 0.502, 0.5, 0.498]  # 127/255 ≈ 0.498; 128/255 ≈ 0.502; also try 0.5
for r in radius_values:
    for ct in contrast_threshold_values:
        for gthresh in gthresh_values:
            dice_scores = []
            iou_scores = []

            for patch_num in range(1, 11):
                gt_dirpath = f'./ear_vessel_data/test/groundtruth/patch_{patch_num}'
                img_dirpath = f'./ear_vessel_data/test/image/patch_{patch_num}'
                file_list = os.listdir(gt_dirpath)

                for image_num in range(1, len(file_list)+1):
                    img_path = f"{image_num}.png"
                    img = imread(os.path.join(img_dirpath, img_path), as_gray=True) / 255.0
                    gt = imread(os.path.join(gt_dirpath, img_path), as_gray=True) / 255
                    gt = (gt > 0.5).astype(np.uint8)

                    # Apply Sauvola threshold
                    thresh = bernsen(img, radius=r, contrast_threshold=ct, gthresh=gthresh)
                    binary = (img > thresh).astype(np.uint8)

                    # Evaluate
                    dice = dice_score(binary, gt)
                    iou = iou_score(binary, gt)

                    dice_scores.append(dice)
                    iou_scores.append(iou)

            results.append({
                'radius': r,
                'contrast_threshold': ct,
                'gthresh': gthresh,
                'mean_dice': np.mean(dice_scores),
                'mean_iou': np.mean(iou_scores)
            })

# Sort and print top configs
top = sorted(results, key=lambda x: -x['mean_dice'])[:10]
for r in top:
    print(f"Radius={r['radius']}, contrast_threshold={r['contrast_threshold']}, gthresh={r['gthresh']}, -> Dice={r['mean_dice']:.4f}, IoU={r['mean_iou']:.4f}")